In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
import clahe

I0000 00:00:1789095614.789262    1605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789095615.996396    1605 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789095619.284750    1605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical/X-Ray/elbow/Train"
testing_path = "../Datasets/data/test/Medical/X-Ray/elbow/Test"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/elbow/valid"

In [3]:
train_dataaug = ImageDataGenerator(
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
    horizontal_flip=True,
    preprocessing_function=lambda img: clahe.clahe_preprocess(img,preprocess_input)
)
testing_dataaug = ImageDataGenerator(preprocessing_function=lambda img : clahe.clahe_preprocess(img,preprocess_input))
valid_dataaug = ImageDataGenerator(preprocessing_function=lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [4]:
train_set = train_dataaug.flow_from_directory(
    training_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary"
)

valid_set = valid_dataaug.flow_from_directory(
    valid_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary",
    shuffle=False
)

test_set = testing_dataaug.flow_from_directory(
    testing_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary",
    shuffle=False
)

Found 4678 images belonging to 2 classes.
Found 576 images belonging to 2 classes.
Found 811 images belonging to 2 classes.


In [5]:
base_model = MobileNetV3Large(weights="imagenet",include_top=False,input_shape=(224,224,3),pooling="avg")

I0000 00:00:1789095626.510390    1605 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
base_model.trainable = False

In [7]:
x = base_model.output
x = Dense(256,activation="relu")(x)
output = Dense(1,activation="sigmoid")(x)

In [8]:
model = Model(inputs = base_model.input,outputs = output)

In [9]:
model.compile(loss="binary_crossentropy",optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [10]:
class_weight = {0: 1.6,1: 1}

In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights= True
)

In [12]:
history1 = model.fit(train_set,validation_data=valid_set,epochs=20,callbacks=early_stop)  

Epoch 1/20


I0000 00:00:1789095630.223304    1605 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789095635.518483    2373 service.cc:153] XLA service 0x7dfaf4033740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789095635.518536    2373 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1789095635.718589    2373 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789095637.187958    2373 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1789095637.238466    2373 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_9824__.162
I0000 00:00:1789095653.415163    2373 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


116/293 ━━━━━━━━━━━━━━━━━━━━ 39s 221ms/step - accuracy: 0.6950 - loss: 0.6200

I0000 00:00:1789095680.722979    2370 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_9824__.162


293/293 ━━━━━━━━━━━━━━━━━━━━ 110s 296ms/step - accuracy: 0.7610 - loss: 0.5064 - val_accuracy: 0.7396 - val_loss: 0.4724
Epoch 2/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 57s 193ms/step - accuracy: 0.8145 - loss: 0.4240 - val_accuracy: 0.7691 - val_loss: 0.4892
Epoch 3/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 58s 197ms/step - accuracy: 0.8260 - loss: 0.3997 - val_accuracy: 0.7795 - val_loss: 0.5278
Epoch 4/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 55s 187ms/step - accuracy: 0.8296 - loss: 0.3867 - val_accuracy: 0.8003 - val_loss: 0.4601
Epoch 5/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 60s 206ms/step - accuracy: 0.8418 - loss: 0.3677 - val_accuracy: 0.7604 - val_loss: 0.6019
Epoch 6/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 60s 204ms/step - accuracy: 0.8540 - loss: 0.3520 - val_accuracy: 0.7812 - val_loss: 0.5490
Epoch 7/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 55s 189ms/step - accuracy: 0.8506 - loss: 0.3461 - val_accuracy: 0.7934 - val_loss: 0.5306
Epoch 8/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 55s 187ms/step - accuracy: 0.8608 - loss: 0.3380 - va

In [13]:
res = model.predict(test_set)

51/51 ━━━━━━━━━━━━━━━━━━━━ 27s 409ms/step


In [14]:
y_pred = (res > 0.5).astype(int)

In [15]:
y_true = test_set.classes

In [16]:
print(classification_report(y_true,y_pred))
print(confusion_matrix(y_true,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.76      0.84       395
           1       0.81      0.96      0.88       416

    accuracy                           0.86       811
   macro avg       0.88      0.86      0.86       811
weighted avg       0.88      0.86      0.86       811

[[299  96]
 [ 15 401]]


In [17]:
model1 = model

In [18]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [19]:
model.compile(optimizer=Adam(learning_rate=1e-5),loss="binary_crossentropy",metrics=["accuracy"])

In [20]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [21]:
history = model.fit(train_set,validation_data=valid_set,epochs=20,callbacks=[early_stop, reduce_lr],class_weight=class_weight)

Epoch 1/20


I0000 00:00:1789096177.816567    2373 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_40311__.186


122/293 ━━━━━━━━━━━━━━━━━━━━ 35s 209ms/step - accuracy: 0.7257 - loss: 0.6127

I0000 00:00:1789096210.640335    2369 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_40311__.186


293/293 ━━━━━━━━━━━━━━━━━━━━ 91s 251ms/step - accuracy: 0.7800 - loss: 0.5471 - val_accuracy: 0.7865 - val_loss: 0.5181 - learning_rate: 1.0000e-05
Epoch 2/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 61s 207ms/step - accuracy: 0.8196 - loss: 0.4951 - val_accuracy: 0.7969 - val_loss: 0.5009 - learning_rate: 1.0000e-05
Epoch 3/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 56s 192ms/step - accuracy: 0.8215 - loss: 0.4759 - val_accuracy: 0.8090 - val_loss: 0.4711 - learning_rate: 1.0000e-05
Epoch 4/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 59s 203ms/step - accuracy: 0.8313 - loss: 0.4658 - val_accuracy: 0.8073 - val_loss: 0.4671 - learning_rate: 1.0000e-05
Epoch 5/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 61s 208ms/step - accuracy: 0.8403 - loss: 0.4503 - val_accuracy: 0.8056 - val_loss: 0.4634 - learning_rate: 1.0000e-05
Epoch 6/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 59s 201ms/step - accuracy: 0.8465 - loss: 0.4400 - val_accuracy: 0.8090 - val_loss: 0.4623 - learning_rate: 1.0000e-05
Epoch 7/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 54s 185ms/step - 

In [22]:
result = model.predict(test_set)

51/51 ━━━━━━━━━━━━━━━━━━━━ 12s 156ms/step


In [23]:
result = (result >= 0.5).astype(int)

In [24]:
print(classification_report(y_true,result))
print(confusion_matrix(y_true,result))

              precision    recall  f1-score   support

           0       0.89      0.82      0.85       395
           1       0.84      0.91      0.87       416

    accuracy                           0.86       811
   macro avg       0.87      0.86      0.86       811
weighted avg       0.87      0.86      0.86       811

[[323  72]
 [ 38 378]]


In [26]:
model.save("elbow_x_ray_model.keras")